# A simple transformer training pipeline

Read top to bottom: **load data → tokenize → batch → train → validate → save → predict**.
This notebook fine-tunes a pretrained Hugging Face model with a visible PyTorch loop.
It does not call the project's experiment runners.

We use only **single-target SEntFiN train/validation headlines**, so each headline has
one unambiguous label. This is sentence classification, not company-targeted sentiment.
The existing verified loader protects the reserved test partitions; we never score them.
Validation is used to choose the best epoch, so these are development results.

## Before you start

From the repository root, install the existing locked environment:

```sh
uv sync --locked --extra transformer --group eda
uv run --locked --extra transformer --group eda python -m ipykernel install --user --name finsight --display-name "FinSight"
```

Open this notebook in VS Code or Jupyter and select **FinSight**. The `eda` group
already includes the kernel; no Hugging Face `datasets`, `Trainer`, or new dependency
is needed. Prepared project data and the selected reviewed checkpoint must already
exist locally (see `docs/guides/windows-gpu-setup.md` and `docs/phase-03-transformers/comparisons/transformer-comparison.md`).
No downloads happen here. GPU is recommended; CPU works but can be very slow.
Run locally: source data, notebook outputs, and saved weights must not be published.


## 1. Imports and settings
Change `MODEL_KEY` to `"roberta_base"`, `"deberta_base"`, or `"finbert"`. Start with one epoch to understand the flow; this is not a reproduction of the locked research experiments.


In [ ]:
import hashlib
import json
import os
import random
import subprocess
from collections import Counter
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from finsight.data.benchmark import load_development_records

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "configs/benchmark-freeze.json").exists():
    raise FileNotFoundError(
        "Open the notebook from the repository or notebooks directory."
    )

MODEL_KEY = "roberta_base"
SEED = 42
EPOCHS = 1
BATCH_SIZE = 8
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = ROOT / "data"
LABELS = ["negative", "neutral", "positive"]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
print(f"Model: {MODEL_KEY}; device: {DEVICE}")

## 2. Load the dataset
Reuse the existing checked partitions, rather than randomly splitting again. Display counts only, keeping source headlines out of saved notebook outputs.


In [ ]:
receipt_path = ROOT / "configs/benchmark-freeze.json"
records = load_development_records(DATA_DIR, receipt_path)
rows = [
    row
    for row in records
    if row["corpus"] == "sentfin" and len(row["native_record"]["sentiments"]) == 1
]
train_rows = [row for row in rows if row["partition"] == "train"]
validation_rows = [row for row in rows if row["partition"] == "validation"]
assert train_rows and validation_rows
assert not (
    {row["group_id"] for row in train_rows}
    & {row["group_id"] for row in validation_rows}
)

train_texts = [row["native_record"]["title"] for row in train_rows]
validation_texts = [row["native_record"]["title"] for row in validation_rows]
train_labels = [
    LABEL_TO_ID[row["native_record"]["sentiments"][0]["label"]] for row in train_rows
]
validation_labels = [
    LABEL_TO_ID[row["native_record"]["sentiments"][0]["label"]]
    for row in validation_rows
]
assert set(train_labels) == set(range(3))
print("Training:", len(train_texts), Counter(train_labels))
print("Validation:", len(validation_texts), Counter(validation_labels))

## 3. Establish a baseline
Predict the most common **training** label for every validation headline. The trained model should be compared against this simple reference.


In [ ]:
majority_label = Counter(train_labels).most_common(1)[0][0]
baseline_predictions = [majority_label] * len(validation_labels)
baseline_f1 = f1_score(
    validation_labels,
    baseline_predictions,
    labels=list(range(3)),
    average="macro",
    zero_division=0,
)
print(f"Majority baseline macro F1: {baseline_f1:.4f}")

## 4. Load the tokenizer and model

The tokenizer turns words into token IDs. Hugging Face supplies the model as a
PyTorch module. RoBERTa/DeBERTa start with a new three-class classification head;
FinBERT retains its pretrained sentiment head and original label ordering.
The warning about a newly initialized head for base models is expected.

Only already reviewed local checkpoint files are used. The small checksum loop is
artifact protection, not part of the learning algorithm. Missing files fail here;
use the existing setup guide rather than downloading unreviewed assets.


In [ ]:
if MODEL_KEY == "finbert":
    from finsight.sentiment.finbert import CHECKPOINT_HASHES, MODEL_ID, REVISION

    model_id, revision = MODEL_ID, REVISION
    checkpoint = (
        DATA_DIR
        / "interim/finbert-checkpoint-cache"
        / ("models--" + model_id.replace("/", "--"))
        / "snapshots"
        / revision
    )
    expected_hashes = CHECKPOINT_HASHES
elif MODEL_KEY in {"roberta_base", "deberta_base"}:
    asset = json.loads((ROOT / "configs/transformer-assets.json").read_text())[
        "models"
    ][MODEL_KEY]
    model_id, revision = asset["model_id"], asset["revision"]
    checkpoint = (
        DATA_DIR
        / "interim/transformer-checkpoint-cache"
        / ("models--" + model_id.replace("/", "--"))
        / "snapshots"
        / revision
    )
    expected_hashes = {name: spec["sha256"] for name, spec in asset["files"].items()}
else:
    raise ValueError("Choose roberta_base, deberta_base, or finbert.")

for filename, expected in expected_hashes.items():
    with (checkpoint / filename).open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != expected:
            raise ValueError(f"Checkpoint checksum differs: {filename}")

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, local_files_only=True, trust_remote_code=False, use_fast=False
)
model_options = (
    {}
    if MODEL_KEY == "finbert"
    else {
        "num_labels": 3,
        "id2label": dict(enumerate(LABELS)),
        "label2id": LABEL_TO_ID,
    }
)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    local_files_only=True,
    trust_remote_code=False,
    weights_only=True,
    attn_implementation="eager",
    **model_options,
).to(DEVICE)
model_labels = [model.config.id2label[i].lower() for i in range(3)]
assert set(model_labels) == set(LABELS)
# FinBERT's output order differs; translate our dataset labels to model indices.
dataset_to_model = {i: model_labels.index(label) for i, label in enumerate(LABELS)}
model_to_dataset = {value: key for key, value in dataset_to_model.items()}

## 5. Tokenize and create batches
Tokenization is preprocessing, not model training. Long inputs are truncated to `MAX_LENGTH`; short ones are padded within each batch. Labels use the model’s output order.


In [ ]:
train_examples = list(zip(train_texts, train_labels, strict=True))
validation_examples = list(zip(validation_texts, validation_labels, strict=True))


def collate_batch(examples: list[tuple[str, int]]) -> dict[str, torch.Tensor]:
    texts, labels = zip(*examples, strict=True)
    batch = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    batch["labels"] = torch.tensor(
        [dataset_to_model[label] for label in labels], dtype=torch.long
    )
    return batch


train_loader = DataLoader(
    train_examples,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    collate_fn=collate_batch,
)
validation_loader = DataLoader(
    validation_examples, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch
)
print("Training batches:", len(train_loader))

## 6. Define validation
`eval()` disables training behavior such as dropout. `no_grad()` prevents gradient calculation. This function only predicts; it never updates weights.


In [ ]:
@torch.no_grad()
def evaluate() -> dict:
    model.eval()
    predictions = []
    for batch in validation_loader:
        inputs = {
            key: value.to(DEVICE) for key, value in batch.items() if key != "labels"
        }
        outputs = model(**inputs)
        predictions.extend(
            model_to_dataset[index]
            for index in outputs.logits.argmax(dim=-1).cpu().tolist()
        )
    return {
        "accuracy": float(accuracy_score(validation_labels, predictions)),
        "macro_f1": float(
            f1_score(
                validation_labels,
                predictions,
                labels=list(range(3)),
                average="macro",
                zero_division=0,
            )
        ),
        "predictions": predictions,
    }

## 7. Train the model

This is the actual training loop: forward pass → loss → backward pass → optimizer
step. All model parameters are updated. After each epoch, evaluate on validation
and keep the weights with the highest validation macro F1. This teaching example
uses fixed-rate AdamW, without the research runner's scheduler or early stopping.
Rerun from the beginning for a fresh, reproducible experiment.


In [ ]:
run_dir = (
    DATA_DIR
    / "interim/notebook-training"
    / (datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ") + "-" + MODEL_KEY)
)
run_dir.mkdir(parents=True, exist_ok=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
best_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    sample_count = 0
    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        count = batch["labels"].shape[0]
        loss_sum += outputs.loss.item() * count
        sample_count += count
        if step % 100 == 0:
            print(f"Epoch {epoch}: batch {step}/{len(train_loader)}")

    result = evaluate()
    history.append(
        {
            "epoch": epoch,
            "training_loss": loss_sum / sample_count,
            "validation_accuracy": result["accuracy"],
            "validation_macro_f1": result["macro_f1"],
        }
    )
    print(history[-1])
    if result["macro_f1"] > best_f1:
        best_f1 = result["macro_f1"]
        model.save_pretrained(run_dir / "best-model", safe_serialization=True)
        tokenizer.save_pretrained(run_dir / "best-model")

## 8. Reload the best model and record results
Save aggregate metrics and provenance next to private weights. These exploratory results do not modify the locked evaluation or public reports.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    run_dir / "best-model", local_files_only=True, trust_remote_code=False
).to(DEVICE)
result = evaluate()
print(
    classification_report(
        validation_labels,
        result["predictions"],
        labels=list(range(3)),
        target_names=LABELS,
        zero_division=0,
    )
)
print(f"Macro F1 — majority: {baseline_f1:.4f}; model: {result['macro_f1']:.4f}")


def membership_digest(rows: list[dict]) -> str:
    return hashlib.sha256(
        json.dumps([row["record_id"] for row in rows]).encode()
    ).hexdigest()


metadata = {
    "scope": "Educational SEntFiN development run; no final test evaluation",
    "configuration": {
        "model_key": MODEL_KEY,
        "model_id": model_id,
        "revision": revision,
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,
        "learning_rate": LEARNING_RATE,
        "device": DEVICE,
    },
    "freeze_receipt": json.loads(receipt_path.read_text()),
    "training_membership_sha256": membership_digest(train_rows),
    "validation_membership_sha256": membership_digest(validation_rows),
    "code_revision": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True
    ).strip(),
    "working_tree_dirty": bool(
        subprocess.check_output(
            ["git", "status", "--porcelain"], cwd=ROOT, text=True
        ).strip()
    ),
    "notebook_sha256": hashlib.sha256(
        (ROOT / "notebooks/02_simple_transformer_training.ipynb").read_bytes()
    ).hexdigest(),
    "versions": {
        name: version(name) for name in ("torch", "transformers", "scikit-learn")
    },
    "majority_baseline_macro_f1": float(baseline_f1),
    "best_validation_macro_f1": result["macro_f1"],
    "history": history,
}
(run_dir / "run.json").write_text(
    json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
)
print("Private run saved to:", run_dir)

## 9. Predict on a new example
Inference is separate from fitting. Softmax values are model probabilities, not calibrated confidence or stock forecasts. This example is synthetic.


In [ ]:
@torch.no_grad()
def predict(text: str) -> dict[str, float]:
    model.eval()
    inputs = tokenizer(
        text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
    )
    outputs = model(**{key: value.to(DEVICE) for key, value in inputs.items()})
    probabilities = outputs.logits.softmax(dim=-1)[0].cpu().tolist()
    return {label: probabilities[model_labels.index(label)] for label in LABELS}


predict("The company reported higher profits this quarter.")

## What to remember

- Hugging Face supplies the tokenizer and pretrained PyTorch model.
- PyTorch computes gradients and updates the weights in section 7.
- Training and validation remain separate; final tests are never scored here.
- Base models have newly initialized sentiment heads; FinBERT already has one.
- One seed and a simple majority baseline are useful for learning, not sufficient
  for research claims. These settings differ from the matched multi-seed studies.
- The SEntFiN cohort has limited coverage of the five project companies. This
  notebook does not establish company-specific or market-prediction performance.
- Local noncommercial dataset/model terms still apply; see `docs/datasets.md`.
- Clear all outputs before sharing or committing this notebook; keep data and
  weights private. The notebook is delivered with no stored execution outputs.
